In [1]:
import copy

import numpy

import cloudvolume
import kimimaro
import fastremap

np = numpy

/home/russelt/Dropbox/gitstuff/Coding/axconn_pcg_test/.pixi/envs/py311-conda-lab/lib/python3.11/site-packages/python_jsonschema_objects/classbuilder.py:643: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  if clsdata.get(


In [2]:
import ac_pcg.label
import ac_pcg.chunks
import ac_pcg.skeletons

In [3]:
import gzip
import pathlib
import pickle

def read_gzip_array(fn, preprocess_func=lambda x: x):
    with gzip.open(fn, "rb") as f:
        a = numpy.load(f)
    return preprocess_func(a)

test_data_path = pathlib.Path(
    "/allen/programs/celltypes/workgroups/em-connectomics/russelt/pcg_axconn/test_data_strip/"
)

test_data_labeled_array_path = test_data_path / "H17_x55_S32_230412_Pos42.npy.gz"
test_skels_path = test_data_path / "H17_x55_S32_230412_Pos42.skels.pkl"

In [4]:
labeled_array = read_gzip_array(test_data_labeled_array_path)
with test_skels_path.open(mode="rb") as skels_fobj:
    label_skels = pickle.load(skels_fobj)

In [5]:
%%time
import rtree

skel_id_to_bboxes = {sk_id: cloudvolume.Bbox.from_points(sk.vertices) for sk_id, sk in label_skels.items()}


p = rtree.index.Property()
p.dimension = 3
label_skel_idx = rtree.index.Index(
    ((sk_id, skel_bb.to_list(), label_skels[sk_id]) for sk_id, skel_bb in skel_id_to_bboxes.items()),
    properties=p
)

CPU times: user 4.81 s, sys: 67.9 ms, total: 4.88 s
Wall time: 4.89 s


In [6]:
# TODO concurrent processing, serial vertex assignment.
#   serial chunk processing method below

In [7]:
import time

chunk_size = (128, 128, 128)
chunk_boxes = ac_pcg.chunks.iterate_chunk_slice_boxes(
    labeled_array.shape, chunk_size)

labeler = ac_pcg.label.ChunkLabeler()

output_arr = numpy.empty(labeled_array.shape, dtype=numpy.uint64)

output_skels = copy.deepcopy(label_skels)

tic = time.time()
for chunk_num, chunk_box in enumerate(chunk_boxes):
    chunk_contains_bb = cloudvolume.Bbox(
        chunk_box.bbox.minpt,
        chunk_box.bbox.maxpt - 1
    )
    subvol_arr = labeled_array[chunk_box.bbox.to_slices()]
    skels_indices_tuples = filter(None, (
        ac_pcg.skeletons.bboxed_skel(res.object, chunk_contains_bb)
        for res in label_skel_idx.intersection(
            chunk_contains_bb.to_list(), objects=True
        )
    ))
    try:
        subvol_skels, subvol_skel_indices = zip(*skels_indices_tuples)
    except ValueError:
        continue

    oversegmented_subvol_arr, oversegmented_subvol_skels = kimimaro.utility.oversegment(
        subvol_arr, subvol_skels, downsample=6, progress=False)

    # convert labels to uint64 layer/chunk indices
    lbl_map = {
        input_lbl: labeler.encode_chunk_seg(chunk_box.chunk_idx, input_lbl)
        for input_lbl in fastremap.unique(
            oversegmented_subvol_arr
        )
    }
    relabeled_arr = fastremap.remap(oversegmented_subvol_arr, lbl_map)
    
    output_arr[chunk_box.bbox.to_slices()] = relabeled_arr[...]

    # map new indices to original skel vertices
    for i, (skel, subvol_indices) in enumerate(zip(oversegmented_subvol_skels, subvol_skel_indices)):
        output_skel = output_skels[skel.id]
        skel.segments = fastremap.remap(skel.segments, lbl_map)
        try:
            output_skel.segments[subvol_indices] = skel.segments
        except AttributeError:
            output_skel.add_vertex_attribute(
                "segments",
                numpy.zeros(output_skel.vertices.shape[0], dtype=numpy.uint64)
            )
            output_skel.segments[subvol_indices] = skel.segments
    

    if not chunk_num % 10:
        print(chunk_num, time.time() - tic, flush=True)

90 0.36019253730773926
100 1.0796589851379395
120 3.643078327178955
130 5.288127660751343
140 6.787588357925415
150 8.431560516357422
160 9.9626784324646
170 11.554019212722778
180 13.435505628585815
190 15.467657089233398
200 17.11019277572632
210 18.89068603515625
220 20.96569037437439
230 22.77479910850525
240 24.53635597229004
250 26.415441274642944
260 28.11994218826294
270 30.472135305404663
280 33.1037712097168
290 35.53700113296509
300 38.13653516769409
310 40.689842224121094
320 42.59112882614136
330 44.407799243927
340 46.0159478187561
350 47.5831458568573
360 49.71430158615112
370 51.93910312652588
390 55.569169759750366
400 57.901896953582764
420 61.77189087867737
430 63.62168860435486
450 67.53278231620789
460 69.61924719810486
480 73.21771550178528
490 74.97751450538635
500 76.60184907913208
510 78.22657251358032
520 79.96294069290161
530 81.45858836174011
540 83.32193493843079
550 85.06827926635742
560 86.4681146144867
570 87.85395169258118
580 89.37801957130432
590 90.4

In [8]:
# write out protobuf edges and precomputed array

In [11]:
from ac_pcg.pcgraph.edges import Edges
from ac_pcg.pcgraph.edges import EDGE_TYPES
from ac_pcg.io.edges import put_chunk_edges


In [16]:
# no cross-chunk edges in how these are processed

in_chunk_edges = Edges([], [])
between_chunk_edges = Edges([], [])
cross_chunk_edges = Edges([], [])


In [17]:

edges_d = {
    EDGE_TYPES.in_chunk: in_chunk_edges,
    EDGE_TYPES.between_chunk: between_chunk_edges,
    EDGE_TYPES.cross_chunk: cross_chunk_edges
}

In [19]:
put_chunk_edges("file://my_chunk", (0, 0, 0), edges_d, compression_level=22)

In [31]:
e = output_skel.segments[output_skel.edges]

In [39]:
%%timeit
edges_mask = (e[:, 0] != e[:, 1])
skel_edges = numpy.unique(
    numpy.sort(
        e[edges_mask], axis=1
    ),
    axis=0)


66.8 μs ± 526 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
